# 06 SQL Warehouses and the SQL Editor

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Understand SQL warehouses (serverless, pro, classic), their sizing, scaling and auto-stop, how they differ from notebook compute, and how to use them: the SQL editor, query history and query profiles, and running SQL on a warehouse programmatically with the Statement Execution API.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
SQL warehouses serve most of a company's analytics: dashboards, BI tools such as Power BI and Tableau, ad hoc SQL, and Genie. Data engineers build the Gold tables those warehouses query, and they're often asked to size or troubleshoot warehouses. The exam's platform domain covers when to use a SQL warehouse.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 SQL warehouse types and settings</b><br>
| Type | Start time | Features | Notes |<br>|---|---|---|---|<br>| <b>Serverless</b> | Seconds | Photon, intelligent workload management, predictive I/O, result cache | Recommended. Free Edition has a serverless starter warehouse |<br>| <b>Pro</b> | Minutes | Photon, Unity Catalog, most features | Runs in your cloud account |<br>| <b>Classic</b> | Minutes | Basic Photon SQL | Legacy |<br><br>
| Setting | Meaning |<br>|---|---|<br>| Size (2X-Small … 4X-Large) | Compute per cluster: bigger = faster complex queries |<br>| Scaling (min/max clusters) | More clusters = more <b>concurrent</b> queries (not faster single queries) |<br>| Auto stop | Stop after N idle minutes to save cost |<br>| Channel | Current or preview SQL features |<br><br>
<b>Size for complexity, scale out for concurrency.</b>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A finance dashboard used by 80 people every Monday at 9:00 was slow: queries queued for minutes. Making the warehouse <b>larger</b> didn't help, because each query was simple. Raising <b>max clusters</b> from 1 to 4 let the warehouse scale out for the Monday peak and back in afterwards, and queue times dropped to seconds. A nightly heavy reconciliation query got its own larger warehouse instead.
</div>

## Using the SQL editor (UI walkthrough)

1. Sidebar → **SQL Editor** → choose your SQL warehouse in the top-right compute selector
2. Pick the catalog and schema (`workspace`, `platform`) in the schema browser, or use three-part names
3. Write a query and run it with Ctrl/Cmd + Enter. Results appear below with a visualization option
4. Use `:param` in the query to add a parameter widget
5. Save the query. Saved queries can feed **dashboards** and **alerts**
6. Sidebar → **Query History**: every query with duration, user and warehouse. Click one to open its **query profile**

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Free Edition includes a serverless SQL warehouse (often called "Serverless Starter Warehouse"). It has usage quotas, so keep demo data small and let it auto-stop.
</div>

## Setup: a small Gold table to query

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a <code>sales_daily</code> table that this lesson and the dashboards lesson (07) query.<br><br>
<b>Why it matters</b><br>Warehouses query <b>tables in Unity Catalog</b>. Building clean, documented Gold tables is the data engineer's part of every BI use case.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A table with 90 days × 3 regions × 4 categories of sales, with comments.
</div>

In [0]:
TABLE = "workspace.platform.sales_daily"
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.platform")

In [0]:
from pyspark.sql import functions as F

days = spark.sql("SELECT explode(sequence(date'2024-09-01', date'2024-11-29', interval 1 day)) AS sale_date")
regions = spark.createDataFrame([("North",), ("South",), ("West",)], "region STRING")
categories = spark.createDataFrame([("Electronics",), ("Furniture",), ("Books",), ("Toys",)], "category STRING")

sales = (
    days.crossJoin(regions).crossJoin(categories)
    .withColumn("orders", (F.abs(F.hash("sale_date", "region", "category")) % 50 + 10).cast("int"))
    .withColumn("revenue", F.round(F.col("orders") * (F.abs(F.hash("category")) % 80 + 20) * (1 + F.dayofweek("sale_date") / 20), 2))
)
sales.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE)
spark.sql(f"COMMENT ON TABLE {TABLE} IS 'Daily sales per region and category (Gold). Demo data for lessons 04_06 and 04_07.'")
spark.sql(f"ALTER TABLE {TABLE} ALTER COLUMN revenue COMMENT 'Revenue in USD'")
print("rows:", spark.table(TABLE).count())

## 1. Find your warehouse

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the SQL warehouses with the Databricks SDK and picks one.<br><br>
<b>Why it matters</b><br>Programmatic access to warehouses (for automation, tests, or apps) needs the warehouse ID.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Your warehouse's name, ID, size, state and auto-stop minutes.
</div>

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
warehouses = list(w.warehouses.list())
for wh in warehouses:
    print(f"{wh.name:<35} id={wh.id} size={wh.cluster_size} state={wh.state} auto_stop={wh.auto_stop_mins}m "
          f"clusters={wh.min_num_clusters}-{wh.max_num_clusters} serverless={wh.enable_serverless_compute}")
WAREHOUSE_ID = warehouses[0].id if warehouses else None

## 2. Run SQL on the warehouse from Python

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Executes a parameterized query on the SQL warehouse with the <b>Statement Execution API</b>, and reads the result.<br><br>
<b>Why it matters</b><br>This is how applications, scripts and services query Databricks without Spark: a REST call to a warehouse. It also shows that the warehouse is <b>separate compute</b> from this notebook.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Revenue per region for November 2024, computed by the warehouse.
</div>

In [0]:
from databricks.sdk.service.sql import StatementParameterListItem

resp = w.statement_execution.execute_statement(
    warehouse_id=WAREHOUSE_ID,
    statement=f"""SELECT region, round(sum(revenue), 2) AS revenue
                  FROM {TABLE}
                  WHERE sale_date >= :start AND sale_date < :end
                  GROUP BY region ORDER BY revenue DESC""",
    parameters=[StatementParameterListItem(name="start", value="2024-11-01", type="DATE"),
                StatementParameterListItem(name="end", value="2024-12-01", type="DATE")],
    wait_timeout="30s",
)
print("state:", resp.status.state)
columns = [c.name for c in resp.manifest.schema.columns]
for row in resp.result.data_array:
    print(dict(zip(columns, row)))

## 3. The same query in the notebook

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs the same SQL with the notebook's own compute.<br><br>
<b>Why it matters</b><br>Same SQL, same tables, different compute. Notebooks suit pipelines and exploration. Warehouses suit BI, dashboards and many concurrent SQL users.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The same revenue per region as section 2.
</div>

In [0]:
spark.sql(
    f"""SELECT region, round(sum(revenue), 2) AS revenue
        FROM {TABLE}
        WHERE sale_date >= :start AND sale_date < :end
        GROUP BY region ORDER BY revenue DESC""",
    args={"start": "2024-11-01", "end": "2024-12-01"},
).show()

## 4. Queries worth saving for BI

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the kind of queries that power dashboards: a trend, a breakdown, and a KPI with comparison to the previous period.<br><br>
<b>Why it matters</b><br>Dashboard queries should be simple, fast and run on Gold tables. Window functions and conditional aggregation (lessons 01_19 and 01_12) cover most needs.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A weekly revenue trend, revenue share per category, and a KPI row comparing November with October.
</div>

In [0]:
spark.sql(f"""
    SELECT date_trunc('week', sale_date) AS week, round(sum(revenue), 0) AS revenue
    FROM {TABLE} GROUP BY 1 ORDER BY 1
""").show(5)

spark.sql(f"""
    SELECT category, round(sum(revenue), 0) AS revenue,
           round(100 * sum(revenue) / sum(sum(revenue)) OVER (), 1) AS share_pct
    FROM {TABLE} GROUP BY category ORDER BY revenue DESC
""").show()

spark.sql(f"""
    SELECT round(sum(revenue) FILTER (WHERE sale_date >= '2024-11-01'), 0) AS nov_revenue,
           round(sum(revenue) FILTER (WHERE sale_date >= '2024-10-01' AND sale_date < '2024-11-01'), 0) AS oct_revenue,
           round(100 * (sum(revenue) FILTER (WHERE sale_date >= '2024-11-01')
                 / sum(revenue) FILTER (WHERE sale_date >= '2024-10-01' AND sale_date < '2024-11-01') - 1), 1) AS growth_pct
    FROM {TABLE}
""").show()

## 5. Query history and profiles

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists recent queries on the warehouse with the SDK, with their duration and status.<br><br>
<b>Why it matters</b><br>Query history is where you find slow or failing dashboard queries, who ran them, and on which warehouse. The query profile (click a query in the UI) shows the operators, time and data read, like the Spark UI for SQL (lesson 02_14).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Your most recent queries with status and duration in milliseconds.
</div>

In [0]:
from databricks.sdk.service.sql import QueryFilter

try:
    for q in list(w.query_history.list(filter_by=QueryFilter(warehouse_ids=[WAREHOUSE_ID]), max_results=5).res or []):
        print(f"{q.status} {q.duration}ms  {(q.query_text or '')[:70]!r}")
except Exception as e:
    print("query history not available here:", type(e).__name__)

## Under the hood

```
SQL editor / dashboard / BI tool / Statement Execution API
        │  SQL over HTTPS (JDBC/ODBC for BI tools)
        ▼
SQL warehouse (serverless): 1..N clusters of Photon compute
   - query queue + workload management (scales out when queries queue)
   - result cache: identical queries on unchanged tables return instantly
   - reads Delta tables in Unity Catalog (permissions checked per user)
        │
        ▼
Delta tables (Gold) on cloud storage
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> every query on a warehouse is a separate statement with its own query profile.

**Result cache:** repeating an identical query on unchanged data returns cached results, which is why dashboards refresh instantly. Delta's transaction log tells the warehouse when the data has changed.

In [0]:
spark.table(TABLE).groupBy("region").agg(F.sum("revenue")).explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: dashboard queries queue at peak times</b><br>
Concurrency problem. Raise the max number of clusters (scale out), not the size.<br><br>
<b>⛔ Problem: one complex query is slow</b><br>Size problem, or an inefficient query. Check its profile, then consider a larger size or a better Gold table (pre-aggregated, clustered).<br><br>
<b>⛔ Problem: warehouse costs keep running overnight</b><br>Auto-stop is too long, or a tool keeps polling. Lower auto-stop and check query history for the source.<br><br>
**⛔ Problem: <code>TABLE_OR_VIEW_NOT_FOUND</code> in the SQL editor**<br>Wrong catalog or schema selected. Use three-part names.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is a Databricks SQL warehouse?</b><br>
Compute dedicated to SQL workloads: the SQL editor, dashboards, alerts, Genie and BI tools via JDBC/ODBC. It runs Photon, scales out for concurrency, auto-stops when idle, and serverless warehouses start in seconds.<br><br>

<b>🎤 2. How do you size a SQL warehouse?</b><br>
Choose the size (T-shirt) for query complexity and data volume, and the min/max number of clusters for concurrency. If queries queue, add clusters. If single queries are slow, check the query first, then increase the size.<br><br>

<b>🎤 3. Serverless vs pro vs classic warehouses?</b><br>
Serverless is managed by Databricks, starts in seconds and has the most features. Pro and classic run in your cloud account and start in minutes. Classic has the fewest features.<br><br>

<b>🎤 4. When would you use a SQL warehouse instead of a notebook on a cluster?</b><br>
For SQL-only analytics: dashboards, BI tools, ad hoc SQL and Genie, especially with many concurrent users. Notebooks are for pipelines, Python work and exploration.<br><br>

<b>🎤 5. How can applications query Databricks without Spark?</b><br>
Through a SQL warehouse, with the Statement Execution REST API, the Databricks SQL connectors (Python, JDBC/ODBC), or the SDKs.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A SQL warehouse serving a dashboard has many queries waiting in a queue during peak hours, but each query runs quickly once started. What should be changed?</b><br>
A. Increase the warehouse size<br>
B. Increase the maximum number of clusters (scaling)<br>
C. Decrease auto-stop<br>
D. Switch to a job cluster<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Scaling out handles concurrency. Size helps individual complex queries.</details><br><br>

<b>Q2. Which Databricks compute is designed to serve BI tools via JDBC/ODBC?</b><br>
A. Job cluster<br>
B. SQL warehouse<br>
C. Instance pool<br>
D. Serverless notebook compute<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>In the SQL editor, write a query on <code>sales_daily</code> with a <code>:region</code> parameter that returns daily revenue for that region, and save it</li><li>Run it for each region and compare the durations in Query History</li><li>From this notebook, run a query on the warehouse with the Statement Execution API that returns the top category per region</li><li>Write down your warehouse's size, scaling and auto-stop, and say whether you'd change them for a 50-user dashboard</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. The saved query (paste into the SQL editor):
print("""SELECT sale_date, round(sum(revenue), 2) AS revenue
FROM workspace.platform.sales_daily
WHERE region = :region
GROUP BY sale_date ORDER BY sale_date""")

# 3. Top category per region (shown here with the notebook's compute; the same SQL runs on the warehouse)
top_sql = f"""
SELECT region, category, revenue FROM (
  SELECT region, category, round(sum(revenue), 0) AS revenue,
         row_number() OVER (PARTITION BY region ORDER BY sum(revenue) DESC) AS rn
  FROM {TABLE} GROUP BY region, category)
WHERE rn = 1 ORDER BY region"""
spark.sql(top_sql).show()

In [0]:
# 3. On the warehouse with the Statement Execution API
resp = w.statement_execution.execute_statement(warehouse_id=WAREHOUSE_ID, statement=top_sql, wait_timeout="30s")
print(resp.result.data_array)

# 4. For a 50-user dashboard: keep a small size, raise max clusters to 2-4 for concurrency, auto-stop 10 minutes.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>SQL warehouses run SQL for the editor, dashboards, alerts, Genie and BI tools</li><li>Serverless (seconds to start, recommended), pro, classic</li><li>Size for query complexity, scale out (clusters) for concurrency, always set auto-stop</li><li>Query history and query profiles show slow queries and why</li><li>Statement Execution API / SQL connectors let apps query a warehouse without Spark</li><li>Data engineers provide clean, documented Gold tables for warehouses to query</li></ul>
</div>

| Task | Where / code |
|---|---|
| Run SQL interactively | SQL Editor → choose warehouse |
| Find slow queries | Query History → query profile |
| List warehouses | `WorkspaceClient().warehouses.list()` |
| Run SQL from code | `w.statement_execution.execute_statement(warehouse_id=..., statement=...)` |
| Parameter in SQL editor | `WHERE region = :region` |

## Git commit

```
git add 04_databricks/06_sql_warehouse.ipynb
git commit -m "add 04_06 sql warehouse notebook"
```